# Stability checks and the 1:20 undersampling ablation (Phase 4)

Three questions about the 38-input LightGBM (28 store features + 8 `roll4` + `weeks_since_open` + `app_score`), all on
the feature lab's 3 temporal folds (train months 0–1 → validate 2, 0–2 → 3, 0–3 → 4). Month 5 and the test months 6–7
are never read.

| § | Question | Why it matters |
|---|---|---|
| 2 | **Reproducibility.** The same default model scored weekly recall 0.4086 (26 Oct), 0.4023 (28 Oct) and 0.4123 (notebook 08). Is the model itself random? | If the same code on the same machine gives different numbers, every comparison so far is shaky |
| 3 | **Seed spread.** How much do the measures move when only LightGBM's random seed changes? | Tells us whether one seed is enough for the final model, and whether tuning's gain is bigger than chance |
| 4 | **1:20 undersampling.** Train on all mules and a random sample of whole normal accounts (about 1 in 25), without class weights, instead of all accounts with class weights | Promised in the 9 Oct plan (professor comment 1 on imbalance): an ablation for the report |

**Measures** (all from `src/models/feature_lab.py`, reported per fold and as the mean of the 3 folds):

- `weekly_recall`: each week, flag the top 1% of open accounts; share of that week's mules among them; mean over weeks 0–26 (the tuning measure).
- `catch_0.1pct`: flags are **permanent** and limited to 0.1% of open accounts per week; share of the accounts that are mules at week *w* already flagged by week *w*, averaged over weeks 0–26.
- `catch_rules`: the same with as many flags per week as the bank's rules raised alerts that week (~6 a week per month): the comparison the project cares about.
- `first_caught_2wk`: with permanent top-1% flags, share of launderers starting in weeks 0–25 first flagged no later than the week after their start.
- `start_recall`: share of mule weeks in the start week and the next that are in the weekly top 1%.

Known refit noise from 5 placebo groups (lab trial `noise-placebo5`, `feature_lab.NOISE`): `catch_0.1pct` 0.005,
`catch_rules` 0.012.

**Caution about the data copy.** The numbers come from this Mac's copy of `data/v1_full`. Its `run_manifest.json` hash
differs from the copies the frozen models were trained on (section 1 prints all three), so absolute numbers are for
comparison inside this notebook only.

## 0. Settings and setup

In [1]:
import os
DATA = os.environ.get("EDA_DATA", "data/v1_full")   # the main dataset; "data/v1" (50k) for a quick dry run
SEEDS = [1, 2, 3, 4, 5]                             # LightGBM random_state values for sections 3 and 4
RATIO = 20                                          # section 4: label-0 rows per mule row in the training sample
SAMPLE_SEED = 42                                    # section 4: which normal accounts are drawn

In [2]:
from __future__ import annotations
import datetime, json, platform, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()                          # find the repo root so Run All works from notebooks/
while not (ROOT / "src" / "amldata").is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("start this notebook from inside the aml-detection-system repo")
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import lightgbm as lgb
import sklearn
from features import manifest
import models.feature_lab as fl
import models.lgbm_baseline as base
import models.stability as st

RUN = os.path.basename(os.path.normpath(DATA))
REPORT = Path(os.environ.get("EDA_REPORT", f"outputs/modelling/models/{RUN}/stability"))
REPORT.mkdir(parents=True, exist_ok=True)
CACHE = REPORT / "runs.csv"           # every fold of every fit in sections 3-4, saved as it ends (a rerun skips them)
show = display if "ipykernel" in sys.modules else print
pd.set_option("display.width", 200)
T0 = time.time()
MACHINE = f"{platform.platform()}, {os.cpu_count()} cores"
print(f"data {DATA} -> {REPORT} | lightgbm {lgb.__version__}, scikit-learn {sklearn.__version__}, numpy {np.__version__}")
print(MACHINE)

data /Users/cheepheng/aml-detection-system/data/v1_full -> outputs/modelling/models/v1_full/stability | lightgbm 4.7.0, scikit-learn 1.9.1, numpy 2.5.3
macOS-26.5.1-arm64-arm-64bit, 15 cores


## 1. The table, the app score and the tuned settings

Built exactly as the feature lab builds its baseline (`stability.lab_setup` calls the lab's own functions): one row
per account per open week 0–26, accounts opened in months 0–4, weeks after the bank closed an account dropped, label
`mule_by_day`; the `roll4` features added at load time; per fold, the application score from a logistic regression
fitted on the training months (out of fold for training accounts, StratifiedKFold seed 42).

The tuned settings are tuning step 2's winner (trial 24 of notebook 08, with `subsample_freq = 1`), read from
`lgbm_tuning_step2/tuned_params.json` and checked against the frozen tuned model's manifest.

In [3]:
t = time.time()
setup = st.lab_setup(DATA, seed=42)
df = setup["df"]
print(f"{len(df):,} account-weeks, {int(df.mule_by_day.sum()):,} mule weeks, {df.account_id.nunique():,} accounts "
      f"({time.time() - t:.0f}s)")

TUNED = json.loads((ROOT / "outputs/modelling/models/v1_full/lgbm_tuning_step2/tuned_params.json").read_text())
frozen_tuned = json.loads((ROOT / "outputs/modelling/models/v1_full/frozen_tuned/manifest.json").read_text())
frozen_default = json.loads((ROOT / "outputs/modelling/models/v1_full/frozen/manifest.json").read_text())
assert {k: frozen_tuned["lightgbm_params"][k] for k in TUNED} == TUNED, "tuned settings differ from the frozen model"
assert frozen_tuned["inputs"] == st.INPUTS and len(st.INPUTS) == 38
print("tuned settings:", TUNED)

here = {"run_manifest_sha256": manifest._sha256(os.path.join(DATA, "run_manifest.json")),
        "preprocess_manifest_sha256": manifest._sha256(os.path.join(DATA, "features", "preprocess_manifest.json"))}
copies = pd.DataFrame([
    {"copy": "this run (this Mac)", **here, "rows": len(df), "mule_rows": int(df.mule_by_day.sum())},
    *[{"copy": f"models/{m['name']} (commit {m['git_commit'][:7]})", "run_manifest_sha256": m["run_manifest_sha256"],
       "preprocess_manifest_sha256": m["preprocess_manifest_sha256"], "rows": m["rows"], "mule_rows": m["mule_rows"]}
      for m in (frozen_default, frozen_tuned)]])
copies["run_manifest_sha256"] = copies["run_manifest_sha256"].str[:12]
copies["preprocess_manifest_sha256"] = copies["preprocess_manifest_sha256"].str[:12]
show(copies)

4,449,169 account-weeks, 8,327 mule weeks, 164,819 accounts (12s)
tuned settings: {'learning_rate': 0.032735723260064485, 'n_estimators': 500, 'num_leaves': 15, 'min_child_samples': 2000, 'colsample_bytree': 0.9047505230698577, 'subsample': 0.918855052953664, 'reg_alpha': 3.785717387766898e-05, 'reg_lambda': 0.00045683792971933475, 'subsample_freq': 1}


,copy,run_manifest_sha256,preprocess_manifest_sha256,rows,mule_rows
0,this run (this Mac),4ce1c4fb593a,5c65b205d31a,4449169,8327
1,models/frozen (commit 8871f4d),1c08c3e81fd6,70188eeef637,4449169,8327
2,models/frozen_tuned (commit fb47487),c5aa290d0721,0a7cb1aafb22,4449169,8327


## 2. Experiment 1: is the default model reproducible on one machine?

The lab's default baseline (LightGBM defaults, class weights `balanced`, seed 42, the 38 inputs) is fitted **twice in
a row** with `feature_lab.fit_folds`, unchanged, and the two results are compared to the last decimal: every measure
and every validation score.

**Rule fixed before the run:** "reproducible on one machine" only if both runs match to 1e-9 on every fold.

In [4]:
def lab_baseline(scores=None):
    t = time.time()
    r, oof = fl.fit_folds(df, st.INPUTS, setup["prof"], setup["start"], scores or setup["scores"], 42, setup["ctx"])
    return r, oof, time.time() - t

run_a, oof_a, sec_a = lab_baseline()
run_b, oof_b, sec_b = lab_baseline()
num = [c for c in run_a.columns if pd.api.types.is_numeric_dtype(run_a[c]) and c != "fold"]
diff_measures = float((run_a[num] - run_b[num]).abs().max().max())
diff_scores = float(np.abs(oof_a["score"].to_numpy() - oof_b["score"].to_numpy()).max())
same_text = run_a.drop(columns=num).equals(run_b.drop(columns=num))
REPRODUCIBLE = bool(diff_measures <= 1e-9 and diff_scores <= 1e-9 and same_text)
print(f"run A {sec_a:.0f}s, run B {sec_b:.0f}s")
print(f"largest difference: measures {diff_measures:.3g}, validation scores {diff_scores:.3g}, "
      f"text columns identical: {same_text}")
print("VERDICT:", "reproducible on one machine (all folds match to 1e-9)" if REPRODUCIBLE else "NOT reproducible")
ab = pd.concat({"run A": run_a.set_index("fold")[st.MEASURES[:5]],
                "run B": run_b.set_index("fold")[st.MEASURES[:5]]}, axis=1).T
show(ab.style.format("{:.10f}") if "ipykernel" in sys.modules else ab)

run A 29s, run B 29s
largest difference: measures 0, validation scores 0, text columns identical: True
VERDICT: reproducible on one machine (all folds match to 1e-9)


### Where did 0.4086, 0.4023 and 0.4123 come from?

All three are the same model (store + `roll4` + app score, LightGBM defaults, seed 42), run on three machines, each
with its own copy of `data/v1_full`. The committed records say who ran what:

| number | where it is recorded | code | who / machine | data copy |
|---|---|---|---|---|
| 0.4086 | lab trials of 26–27 Oct (`agent-trials-phases`, `max-out-vs-hist`, `internal-out-share`) | `bf43eac` / `46e3d35` (+ uncommitted) | Kee Chee Pheng, this Mac | this Mac's (`run_manifest` 4ce1c4…, the hash in `outputs/modelling/preprocess/params.json`) |
| 0.4023 | lab trials of 28 Oct (`noise-placebo5`, `graph`, `app-split`) | `dd63d9e` + uncommitted lab changes (committed as `8871f4d`) | Emmy Li | most likely the copy of `models/frozen` (1c08c3…, trained from `8871f4d`) |
| 0.4123 | notebook 08, trial 0 and section 4 | `fb47487` | chailiam1000-design, Windows (paths printed with `\`) | the copy of `models/frozen_tuned` (c5aa29…) |

Below, the per-fold numbers of each record next to this machine's run, then checks that separate the explanations:
(C) the same model through this notebook's own loop, (D) notebook 08's trial-0 settings (`reg_alpha` and `reg_lambda`
1e-8 and `subsample_freq = 1` instead of the plain defaults), and (E, F) the app score nudged by one part in a billion
and one part in a million, the size of difference another machine's maths library can leave in a logistic regression.

In [5]:
lab_root = ROOT / "outputs/modelling/feature_lab/v1_full"
def lab_record(trial):
    f = pd.read_csv(lab_root / trial / "folds.csv")
    return f[(f.variant == "baseline") & (f.view == "app")].set_index("fold")["weekly_recall"]
nb08 = pd.read_csv(ROOT / "outputs/modelling/models/v1_full/lgbm_tuning_step2/trials.csv").set_index("trial")
records = pd.DataFrame({
    "26 Oct, this Mac (agent-trials-phases)": lab_record("agent-trials-phases"),
    "27 Oct, this Mac (max-out-vs-hist)": lab_record("max-out-vs-hist"),
    "28 Oct, Emmy (noise-placebo5)": lab_record("noise-placebo5"),
    "28 Oct, Emmy (graph)": lab_record("graph"),
    "28 Oct, Windows (notebook 08 trial 0)": pd.Series({f: nb08.loc[0, f"fold{f}_weekly_recall"] for f in (1, 2, 3)}),
    "today, this Mac, run A": run_a.set_index("fold")["weekly_recall"],
}).T
records["mean"] = records[[1, 2, 3]].mean(axis=1)
show(records.style.format("{:.10f}") if "ipykernel" in sys.modules else records)
SAME_AS_26OCT = bool((records.loc["today, this Mac, run A", [1, 2, 3]]
                      - records.loc["26 Oct, this Mac (agent-trials-phases)", [1, 2, 3]]).abs().max() <= 1e-9)
print("this Mac today reproduces this Mac on 26-27 Oct (to 1e-9):", SAME_AS_26OCT)

,1,2,3,mean
"26 Oct, this Mac (agent-trials-phases)",0.4069394260,0.3661459650,0.4526484759,0.4085779556
"27 Oct, this Mac (max-out-vs-hist)",0.4069394260,0.3661459650,0.4526484759,0.4085779556
"28 Oct, Emmy (noise-placebo5)",0.4195217780,0.3540945551,0.4333541366,0.4023234899
"28 Oct, Emmy (graph)",0.4195217780,0.3540945551,0.4333541366,0.4023234899
"28 Oct, Windows (notebook 08 trial 0)",0.4113656119,0.3786446757,0.4468887421,0.4122996766
"today, this Mac, run A",0.4069394260,0.3661459650,0.4526484759,0.4085779556


this Mac today reproduces this Mac on 26-27 Oct (to 1e-9): True


In [6]:
checks = {"C: same model, this notebook's loop": st.fit_eval(setup, params=None, seed=42)}
NB08_TRIAL0 = {"learning_rate": 0.1, "n_estimators": 100, "num_leaves": 31, "min_child_samples": 20,
               "colsample_bytree": 1.0, "subsample": 1.0, "reg_alpha": 1e-8, "reg_lambda": 1e-8, "subsample_freq": 1}
checks["D: notebook 08 trial-0 settings"] = st.fit_eval(setup, params=NB08_TRIAL0, seed=42)
for label, rel in (("E: app score x (1 + 1e-9 noise)", 1e-9), ("F: app score x (1 + 1e-6 noise)", 1e-6)):
    checks[label] = st.fit_eval(setup, seed=42, scores=st.perturb_scores(setup["scores"], rel))

rows = []
for label, (r, oof) in {"A: lab baseline": (run_a, oof_a), "B: lab baseline again": (run_b, oof_b), **checks}.items():
    w = r.set_index("fold")
    rows.append({"run": label, **{f"weekly_recall f{f}": w.loc[f, "weekly_recall"] for f in (1, 2, 3)},
                 "weekly_recall mean": w["weekly_recall"].mean(), "catch_rules mean": w["catch_rules"].mean(),
                 "catch_0.1pct mean": w["catch_0.1pct"].mean(),
                 "max |score - A|": float(np.abs(oof["score"].to_numpy() - oof_a["score"].to_numpy()).max())})
exp1 = pd.DataFrame(rows).set_index("run")
show(exp1.style.format("{:.6g}") if "ipykernel" in sys.modules else exp1)
assert exp1.loc["C: same model, this notebook's loop", "max |score - A|"] == 0, \
    "stability.fit_eval does not reproduce feature_lab.fit_folds"
exp1_folds = pd.concat([r.assign(run=k) for k, (r, _) in
                        {"A: lab baseline": (run_a, None), "B: lab baseline again": (run_b, None), **checks}.items()],
                       ignore_index=True)
del oof_b, checks

,weekly_recall f1,weekly_recall f2,weekly_recall f3,weekly_recall mean,catch_rules mean,catch_0.1pct mean,max |score - A|
run,,,,,,,
A: lab baseline,0.406939,0.366146,0.452648,0.408578,0.452358,0.556883,0
B: lab baseline again,0.406939,0.366146,0.452648,0.408578,0.452358,0.556883,0
"C: same model, this notebook's loop",0.406939,0.366146,0.452648,0.408578,0.452358,0.556883,0
D: notebook 08 trial-0 settings,0.406939,0.366146,0.452648,0.408578,0.452358,0.556883,4.66552e-10
E: app score x (1 + 1e-9 noise),0.406939,0.366146,0.452648,0.408578,0.452358,0.556883,0
F: app score x (1 + 1e-6 noise),0.425626,0.358451,0.452648,0.412242,0.453142,0.553208,0.69569


### What experiment 1 shows

- **The model is reproducible on one machine.** Two runs in a row gave identical measures and identical validation
  scores (largest difference 0, the rule needed 1e-9 or less). The same holds across days: today's run on this Mac equals
  this Mac's lab runs of 26 and 27 Oct to the last decimal (0.4069 / 0.3661 / 0.4526, mean 0.4086), although the code
  moved on by several commits in between.
- **The settings are not the cause.** Notebook 08's trial-0 settings (penalties 1e-8, `subsample_freq = 1`) give the
  same measures as the plain defaults (scores differ by at most 5e-10).
- **Tiny number differences are enough.** Changing the app score by one part in a billion changes nothing, but by one
  part in a million (check F) moves weekly recall per fold by +0.019 / −0.008 / 0.000 and the mean from 0.4086 to
  0.4122. That is as large as the gaps between the three recorded numbers (0.4023 to 0.4123).
- **Most likely cause of 0.4086 / 0.4023 / 0.4123:** each number was produced on a different computer from a different
  copy of `data/v1_full` (three different `run_manifest.json` hashes; Mac, Emmy's machine, Windows). Somewhere between the
  data, the cleaned tables and the app score's logistic regression (different maths libraries round the last digits
  differently), the inputs differ in their last digits, and LightGBM turns such differences into a different set of trees.
  The code, the settings, the row order and the seeds were the same in all three runs. We cannot say which of data or
  maths library it is without the other two machines' files (the manifests' hashes also cover file paths, git commits
  and package versions, so they differ even for identical data).
- **How to check and fix (not done here):** (1) each machine prints a hash of `weekly.parquet`, `accounts_linear.parquet`
  and of the fold-1 app scores (`fl.app_scores(out, 42)[1]`), so the team sees where the copies first differ; (2) all
  reported numbers come from one machine and one data copy (the Phase 5 harness run); (3) a gap smaller than the seed
  spread in section 3 is treated as no difference.

## 3. Experiment 2: how much does LightGBM's seed move the tuned model?

The tuned settings are fitted with `random_state` = 1, 2, 3, 4, 5; data and app score unchanged. The seed decides
which rows each tree sees (`subsample` 0.92), which inputs (`colsample_bytree` 0.90), and which 200,000 rows LightGBM
samples to place its histogram bins. Seed 42 (the frozen model's) is added for reference, and the **default** settings
with the same 5 seeds, so the tuned model's lead over the default is judged against chance on both sides.

**Rule fixed before the run:** if the SD across seeds 1–5 (ddof = 1) of `catch_0.1pct` exceeds 0.005 **or** of
`catch_rules` exceeds 0.012 in any fold, recommend averaging the 5 seeds' scores in the final model; otherwise a
single seed.

In [7]:
def cached_run(variant: str, seed: int, **kw) -> pd.DataFrame:
    # fit (or load from runs.csv, if an earlier execution finished it) one variant with one seed on the 3 folds
    done = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame(columns=["variant", "seed"])
    hit = done[(done["variant"] == variant) & (done["seed"] == seed)]
    if len(hit) == 3:
        return hit.reset_index(drop=True)
    t = time.time()
    r, _ = st.fit_eval(setup, seed=seed, **kw)
    r = r.assign(variant=variant, seed=seed, run_seconds=time.time() - t,
                 fitted=datetime.datetime.now().isoformat(timespec="seconds"))
    pd.concat([done, r], ignore_index=True).to_csv(CACHE, index=False)
    print(f"{variant:20s} seed {seed:2d}: weekly recall {r.weekly_recall.mean():.4f}  catch_rules "
          f"{r.catch_rules.mean():.4f}  catch_0.1pct {r['catch_0.1pct'].mean():.4f}  ({time.time() - t:.0f}s)")
    return r

runs = []
for s in SEEDS + [42]:
    runs.append(cached_run("tuned", s, params=TUNED))
for s in SEEDS:
    runs.append(cached_run("default", s))
runs = pd.concat(runs + [run_a.assign(variant="default", seed=42)], ignore_index=True)   # + Experiment 1's run A

tuned                seed  1: weekly recall 0.4282  catch_rules 0.4463  catch_0.1pct 0.5610  (52s)


tuned                seed  2: weekly recall 0.4311  catch_rules 0.4611  catch_0.1pct 0.5584  (55s)


tuned                seed  3: weekly recall 0.4336  catch_rules 0.4510  catch_0.1pct 0.5603  (613s)


tuned                seed  4: weekly recall 0.4370  catch_rules 0.4576  catch_0.1pct 0.5601  (302s)


tuned                seed  5: weekly recall 0.4320  catch_rules 0.4499  catch_0.1pct 0.5595  (55s)


tuned                seed 42: weekly recall 0.4303  catch_rules 0.4534  catch_0.1pct 0.5653  (257s)


default              seed  1: weekly recall 0.4009  catch_rules 0.4557  catch_0.1pct 0.5549  (27s)


default              seed  2: weekly recall 0.4087  catch_rules 0.4552  catch_0.1pct 0.5542  (1082s)


default              seed  3: weekly recall 0.4072  catch_rules 0.4593  catch_0.1pct 0.5636  (26s)


default              seed  4: weekly recall 0.4128  catch_rules 0.4502  catch_0.1pct 0.5602  (27s)


default              seed  5: weekly recall 0.4118  catch_rules 0.4607  catch_0.1pct 0.5651  (27s)


In [8]:
SPREAD_MEASURES = ["weekly_recall", "catch_0.1pct", "catch_rules", "first_caught_2wk", "start_recall"]
spread = st.seed_summary(runs[runs.seed.isin(SEEDS)], SPREAD_MEASURES)
show(spread.set_index(["variant", "fold"])[[f"{m}_{s}" for m in SPREAD_MEASURES for s in ("mean", "sd")]].round(4))

tuned5 = spread[spread.variant == "tuned"].set_index("fold")
default5 = spread[spread.variant == "default"].set_index("fold")
LIMITS = {"catch_0.1pct": fl.NOISE["catch_0.1pct"], "catch_rules": fl.NOISE["catch_rules"]}
over = {m: [f for f in (1, 2, 3) if tuned5.loc[f, f"{m}_sd"] > lim] for m, lim in LIMITS.items()}
AVERAGE_SEEDS = any(over.values())
for m, lim in LIMITS.items():
    print(f"{m:13s} SD across seeds per fold: " + " / ".join(f"{tuned5.loc[f, f'{m}_sd']:.4f}" for f in (1, 2, 3))
          + f"  (limit {lim}; folds over: {over[m] or 'none'})")
print("RECOMMENDATION:", "average the 5 seeds' scores in the final model" if AVERAGE_SEEDS else "a single seed is enough")

weekly_recall_mean  weekly_recall_sd  catch_0.1pct_mean  catch_0.1pct_sd  catch_rules_mean  catch_rules_sd  first_caught_2wk_mean  first_caught_2wk_sd  start_recall_mean  \
variant fold                                                                                                                                                                              
default 1                 0.4091            0.0033             0.5768           0.0105            0.4806          0.0036                 0.9688               0.0000             0.8252   
        2                 0.3689            0.0077             0.5401           0.0092            0.4569          0.0054                 0.9257               0.0064             0.7145   
        3                 0.4468            0.0087             0.5619           0.0035            0.4311          0.0112                 0.9680               0.0073             0.8199   
tuned   1                 0.4272            0.0041             0.5714           0.0066            0.4698          0.0102                 0.9688               0.0000             0.8362   
        2                 0.4129            0.0082             0.5441           0.0081            0.4528          0.0045                 0.9343               0.0078             0.7667   
        3                 0.4570            0.0046             0.5641           0.0062            0.4369          0.0072                 0.9733               0.0000             0.8278   

              start_recall_sd  
variant fold                   
default 1              0.0066  
        2              0.0132  
        3              0.0109  
tuned   1              0.0103  
        2              0.0130  
        3              0.0000

catch_0.1pct  SD across seeds per fold: 0.0066 / 0.0081 / 0.0062  (limit 0.005; folds over: [1, 2, 3])
catch_rules   SD across seeds per fold: 0.0102 / 0.0045 / 0.0072  (limit 0.012; folds over: none)
RECOMMENDATION: average the 5 seeds' scores in the final model


In [9]:
# the tuned model's lead over the default per fold: means over seeds 1-5, against the seed spread of each
lead_rows = []
for m in SPREAD_MEASURES:
    for f in (1, 2, 3):
        t42 = runs[(runs.variant == "tuned") & (runs.seed == 42) & (runs.fold == f)][m].iloc[0]
        lead_rows.append({"measure": m, "fold": f, "tuned mean": tuned5.loc[f, f"{m}_mean"],
                          "default mean": default5.loc[f, f"{m}_mean"],
                          "lead (means)": tuned5.loc[f, f"{m}_mean"] - default5.loc[f, f"{m}_mean"],
                          "lead (seed 42)": t42 - run_a.set_index("fold").loc[f, m],
                          "SD tuned": tuned5.loc[f, f"{m}_sd"], "SD default": default5.loc[f, f"{m}_sd"]})
lead = pd.DataFrame(lead_rows)
# larger than the seed spread: beyond one seed's SD (either model), and beyond 2 standard errors of the difference
lead["lead > seed SD"] = lead["lead (means)"].abs() > lead[["SD tuned", "SD default"]].max(axis=1)
lead["2 SE of lead"] = 2 * np.sqrt((lead["SD tuned"] ** 2 + lead["SD default"] ** 2) / len(SEEDS))
lead["lead > 2 SE"] = lead["lead (means)"].abs() > lead["2 SE of lead"]
show(lead.round(4))

,measure,fold,tuned mean,default mean,lead (means),lead (seed 42),SD tuned,SD default,lead > seed SD,2 SE of lead,lead > 2 SE
0,weekly_recall,1,0.4272,0.4091,0.0181,0.0222,0.0041,0.0033,True,0.0047,True
1,weekly_recall,2,0.4129,0.3689,0.0440,0.0329,0.0082,0.0077,True,0.0101,True
2,weekly_recall,3,0.4570,0.4468,0.0102,0.0099,0.0046,0.0087,True,0.0088,True
3,catch_0.1pct,1,0.5714,0.5768,-0.0054,0.0017,0.0066,0.0105,False,0.0111,False
4,catch_0.1pct,2,0.5441,0.5401,0.0041,0.0152,0.0081,0.0092,False,0.0109,False
5,catch_0.1pct,3,0.5641,0.5619,0.0022,0.0084,0.0062,0.0035,False,0.0064,False
6,catch_rules,1,0.4698,0.4806,-0.0108,0.0007,0.0102,0.0036,True,0.0097,True
7,catch_rules,2,0.4528,0.4569,-0.0042,-0.0048,0.0045,0.0054,False,0.0063,False
8,catch_rules,3,0.4369,0.4311,0.0058,0.0073,0.0072,0.0112,False,0.0119,False
9,first_caught_2wk,1,0.9688,0.9688,0.0000,0.0000,0.0000,0.0000,False,0.0000,False


### What experiment 2 shows

- **Seed spread of the tuned model (SD over seeds 1–5, per fold):** `catch_0.1pct` 0.0066 / 0.0081 / 0.0062, `catch_rules`
  0.0102 / 0.0045 / 0.0072, weekly recall 0.0041 / 0.0082 / 0.0046.
- **Pre-registered rule:** `catch_0.1pct` is above 0.005 in all 3 folds (`catch_rules` stays under 0.012 in all), so the
  rule says: **average the 5 seeds' scores in the final model.** Not implemented here; `train.py` is unchanged.
- **The tuned model's lead over the default is real on weekly recall, not on the earliness measures.** Weekly recall:
  +0.018 / +0.044 / +0.010, larger than the seed spread and than 2 standard errors in every fold. `catch_0.1pct`:
  −0.005 / +0.004 / +0.002 and `catch_rules`: −0.011 / −0.004 / +0.006, both within the seed spread (in fold 1 the tuned
  model is slightly behind on `catch_rules`). `first_caught_2wk` is flat. Tuning bought re-flags of mules inside the top
  1%, which the tuning measure rewards, but no earlier first flags at the rules' budget.
- **A note on the lab's noise levels.** The placebo noise (`catch_0.1pct` 0.005, `catch_rules` 0.012) was measured with
  one seed. A change of seed alone moves `catch_0.1pct` by about 0.006–0.008 (SD), so a single-seed gain under ~0.015 on
  that measure cannot be told apart from seed luck.

## 4. Experiment 3: 1:20 undersampling instead of class weights

Each fold's training months are resampled; the validation month is untouched.

- Every account that is a mule in at least one training week is kept **whole** (all its weeks, including a recruited
  customer's normal weeks before the start).
- Normal accounts (no mule week) are drawn **whole**, in a random order (seed 42), until the label-0 rows reach
  20 × the mule rows. Single weeks are never sampled.
- LightGBM with the tuned settings and **no class weights** is fitted on that sample, with seeds 1–5, and compared
  with the class-weight tuned model of section 3 with the same seed (paired by seed).

No adoption rule was set in advance: this is an ablation for the report (professor comment 1 on imbalance). The tuned
settings were chosen for the full table; `min_child_samples = 2000` is large for a sample of ~60,000–125,000 rows,
which works against the sample. That is part of the comparison as asked, not retuned.

In [10]:
sample_info = []
for f, (train_m, valid_m) in enumerate(base.FOLDS, start=1):
    tr = df[df.open_month.isin(train_m)]
    s = st.undersample(tr, RATIO, SAMPLE_SEED)
    sample_info.append({"fold": f, "train rows (all)": len(tr), "train rows (sample)": len(s),
                        "mule rows": int(s.mule_by_day.sum()), "label-0 rows (sample)": int((s.mule_by_day == 0).sum()),
                        "label-0 per mule row": (s.mule_by_day == 0).sum() / s.mule_by_day.sum(),
                        "accounts (all)": tr.account_id.nunique(), "accounts (sample)": s.account_id.nunique(),
                        "share of rows kept": len(s) / len(tr)})
sample_info = pd.DataFrame(sample_info)
show(sample_info.round(4))
del tr, s

under = pd.concat([cached_run("tuned, 1:20 sample", s, params=TUNED, class_weight=None, sample_ratio=RATIO,
                              sample_seed=SAMPLE_SEED) for s in SEEDS], ignore_index=True)

,fold,train rows (all),train rows (sample),mule rows,label-0 rows (sample),label-0 per mule row,accounts (all),accounts (sample),share of rows kept
0,1,1708442,73889,3518,70371,20.0031,63293,2754,0.0432
1,2,2613390,104820,4991,99829,20.0018,96816,3906,0.0401
2,3,3604780,137656,6555,131101,20.0002,133540,5128,0.0382


tuned, 1:20 sample   seed  1: weekly recall 0.4259  catch_rules 0.4568  catch_0.1pct 0.5567  (28s)


tuned, 1:20 sample   seed  2: weekly recall 0.4243  catch_rules 0.4545  catch_0.1pct 0.5581  (31s)


tuned, 1:20 sample   seed  3: weekly recall 0.4256  catch_rules 0.4519  catch_0.1pct 0.5588  (137s)


tuned, 1:20 sample   seed  4: weekly recall 0.4231  catch_rules 0.4582  catch_0.1pct 0.5540  (59s)


tuned, 1:20 sample   seed  5: weekly recall 0.4272  catch_rules 0.4610  catch_0.1pct 0.5630  (52s)


In [11]:
EXP3_MEASURES = SPREAD_MEASURES + ["acc_pr_auc", "fit_seconds"]
cw = runs[(runs.variant == "tuned") & runs.seed.isin(SEEDS)].set_index(["seed", "fold"])[EXP3_MEASURES].sort_index()
us = under.set_index(["seed", "fold"])[EXP3_MEASURES].sort_index()
paired = us - cw
exp3 = pd.DataFrame({
    "class weights (mean of seeds)": cw.groupby(level="fold").mean().stack(),
    "1:20 sample (mean of seeds)": us.groupby(level="fold").mean().stack(),
    "paired change (mean)": paired.groupby(level="fold").mean().stack(),
    "paired change (min)": paired.groupby(level="fold").min().stack(),
    "paired change (max)": paired.groupby(level="fold").max().stack()}).rename_axis(["fold", "measure"])
exp3 = exp3.swaplevel().sort_index(level=0, sort_remaining=False)
show(exp3.round(4))

by_seed_cw, by_seed_us = cw.groupby(level="seed").mean(), us.groupby(level="seed").mean()   # mean of the 3 folds
means = pd.DataFrame({"class weights": by_seed_cw.mean(), "1:20 sample": by_seed_us.mean()})
means["change"] = means["1:20 sample"] - means["class weights"]
means["seeds where the sample is higher"] = [int((by_seed_us[m] > by_seed_cw[m]).sum()) for m in means.index]
means["folds where the sample is higher (mean of seeds)"] = [
    int((us.groupby(level="fold").mean()[m] > cw.groupby(level="fold").mean()[m]).sum()) for m in means.index]
show(means.round(4))

class weights (mean of seeds)  1:20 sample (mean of seeds)  paired change (mean)  paired change (min)  paired change (max)
measure          fold                                                                                                                            
acc_pr_auc       1                            0.8709                       0.8474               -0.0235              -0.0304              -0.0173
                 2                            0.8625                       0.8249               -0.0375              -0.0522              -0.0228
                 3                            0.8664                       0.8688                0.0025              -0.0031               0.0054
catch_0.1pct     1                            0.5714                       0.5663               -0.0051              -0.0179               0.0096
                 2                            0.5441                       0.5406               -0.0035              -0.0158               0.0090
                 3                            0.5641                       0.5674                0.0033              -0.0075               0.0169
catch_rules      1                            0.4698                       0.4755                0.0057              -0.0023               0.0198
                 2                            0.4528                       0.4548                0.0020              -0.0172               0.0217
                 3                            0.4369                       0.4392                0.0023              -0.0003               0.0094
first_caught_2wk 1                            0.9688                       0.9781                0.0094               0.0000               0.0156
                 2                            0.9343                       0.9429                0.0086               0.0000               0.0143
                 3                            0.9733                       0.9733                0.0000               0.0000               0.0000
fit_seconds      1                            7.0064                       3.0218               -3.9846              -5.4988              -2.3076
                 2                           73.6346                       4.6726              -68.9620            -318.6055              -4.9141
                 3                           15.3666                       4.5886              -10.7780             -14.8725              -7.0107
start_recall     1                            0.8362                       0.8362               -0.0000              -0.0157               0.0079
                 2                            0.7667                       0.7812                0.0145               0.0072               0.0290
                 3                            0.8278                       0.8265               -0.0013              -0.0066               0.0000
weekly_recall    1                            0.4272                       0.4048               -0.0224              -0.0299              -0.0156
                 2                            0.4129                       0.4176                0.0046              -0.0043               0.0118
                 3                            0.4570                       0.4533               -0.0037              -0.0148               0.0054

,class weights,1:20 sample,change,seeds where the sample is higher,folds where the sample is higher (mean of seeds)
weekly_recall,0.4324,0.4252,-0.0072,0,1
catch_0.1pct,0.5599,0.5581,-0.0017,1,1
catch_rules,0.4532,0.4565,0.0033,4,3
first_caught_2wk,0.9588,0.9648,0.0060,5,2
start_recall,0.8102,0.8146,0.0044,3,1
acc_pr_auc,0.8666,0.8471,-0.0195,0,1
fit_seconds,32.0025,4.0943,-27.9082,0,0


### What experiment 3 shows

- The sample keeps about 4% of the training rows (74k–138k of 1.7M–3.6M), 20.0 label-0 rows per mule row.
- **On the earliness measures the two ways of handling imbalance are equivalent:** `catch_rules` +0.003, `catch_0.1pct`
  −0.002, `first_caught_2wk` +0.006, `start_recall` +0.004 (mean of folds and seeds), all within the seed spread.
  `catch_rules` is higher in all 3 folds, but only by 0.002–0.006, under the lab's 0.012 noise level.
- **On the ranking measures the sample is slightly worse in fold 1:** weekly recall −0.022 in fold 1 (lower for all 5
  seeds), +0.005 and −0.004 in folds 2–3 (mean −0.007); account-level PR-AUC −0.024 / −0.038 / +0.002 (mean −0.020).
  Fold 1 has the smallest training set (3,518 mule rows), where throwing away 96% of normal accounts costs most.
- **Training is 2–4x faster** (median LightGBM fit time per fold 3.1 / 4.2 / 3.6 s vs 6.9 / 10.2 / 16.3 s with class
  weights; other jobs were running on this Mac, so means are inflated and only the medians are compared).
- Plainly: undersampling to 1:20 neither helps nor hurts early detection; class weights rank slightly better when
  training data are scarce. The tuned settings were chosen for the full table and not retuned for the sample.

## 5. Results files and report

In [12]:
SHOW = ["weekly_recall", "catch_0.1pct", "catch_rules", "first_caught_2wk", "start_recall"]
def per_fold_table(r: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    t = r.pivot_table(index=by, columns="fold", values=SHOW)
    t.columns = [f"{m} f{f}" for m, f in t.columns]
    for m in SHOW:
        t[f"{m} mean"] = r.groupby(by)[m].mean()
    return t[[c for m in SHOW for c in (f"{m} f1", f"{m} f2", f"{m} f3", f"{m} mean")]]

all_runs = pd.concat([exp1_folds.assign(experiment=1),
                      runs[(runs.seed != 42) | (runs.variant == "tuned")].assign(experiment=2),
                      under.assign(experiment=3)], ignore_index=True)
front = ["experiment", "run", "variant", "seed", "fold"]
all_runs = all_runs[front + [c for c in all_runs.columns if c not in front + ["fitted", "run_seconds"]]]
all_runs.to_csv(REPORT / "all_folds.csv", index=False)
exp1.to_csv(REPORT / "exp1_reproducibility.csv")
records.to_csv(REPORT / "exp1_recorded_runs.csv")
spread.to_csv(REPORT / "exp2_seed_spread.csv", index=False)
lead.to_csv(REPORT / "exp2_tuned_lead.csv", index=False)
exp3.to_csv(REPORT / "exp3_undersampling.csv")
sample_info.to_csv(REPORT / "exp3_sample.csv", index=False)
N_FITS = all_runs.groupby(["experiment", "run", "variant", "seed"], dropna=False).ngroups
print(f"{N_FITS} model variants fitted and scored on the 3 folds")

def fmt(t, index=True):
    # 4 decimals for measures, whole numbers for counts
    cols = ([t.index] if index else []) + [t[c] for c in t.columns]
    return t.to_markdown(index=index, floatfmt=tuple(".0f" if pd.api.types.is_integer_dtype(c) else ".4f" for c in cols))
lines = [
    "# Stability checks and 1:20 undersampling (notebook 10)", "",
    f"Written by `notebooks/10_stability_undersampling.ipynb` on {datetime.date.today()}, data `{RUN}` (this Mac's copy), "
    f"commit {fl.git_commit()}, {MACHINE}, lightgbm {lgb.__version__}. Months 0-4 only, the lab's 3 temporal folds; "
    "month 5 and test months 6-7 never read.", "",
    f"**Data copy.** This Mac's `data/{RUN}` (`run_manifest.json` sha256 "
    f"{here['run_manifest_sha256'][:12]}...) differs by hash from the copies the frozen models were trained on "
    f"(frozen {frozen_default['run_manifest_sha256'][:12]}..., frozen_tuned {frozen_tuned['run_manifest_sha256'][:12]}...); "
    "the hash also covers the BAF path, git commit and package versions, so it does not by itself prove the data "
    "differ. Absolute numbers are for comparison inside this report only.", "",
    f"**Looks at the {RUN} folds.** {N_FITS} model variants were fitted and scored on the 3 folds here (none is a "
    "feature trial and none was chosen by these numbers; no lab ledger rows were written, because only the lab's own "
    "command appends to it).", "",
    "## Experiment 1: reproducibility", "",
    "Rule (fixed before the run): reproducible on one machine only if two runs match to 1e-9 on every fold. "
    f"Largest difference between two runs of the lab's baseline: measures {diff_measures:.3g}, validation scores "
    f"{diff_scores:.3g}. **{'Reproducible' if REPRODUCIBLE else 'Not reproducible'} on one machine.**", "",
    "Recorded runs of the same default model (weekly recall per fold):", "", records.to_markdown(floatfmt=".6f"), "",
    f"This Mac today reproduces this Mac's 26-27 Oct runs to 1e-9: **{SAME_AS_26OCT}**.", "",
    "Checks (default model unless stated; max |score - A| = largest change in any validation score):", "",
    exp1.to_markdown(floatfmt=".6g"), "",
    "## Experiment 2: seed spread (tuned settings, random_state 1-5; default settings for comparison)", "",
    fmt(per_fold_table(runs, ["variant", "seed"])), "",
    "SD across seeds 1-5 (ddof = 1):", "",
    fmt(spread.set_index(["variant", "fold"])[[f"{m}_sd" for m in SPREAD_MEASURES]]), "",
    "Rule (fixed before the run): average 5 seeds if the SD of catch_0.1pct > 0.005 or of catch_rules > 0.012 in any "
    f"fold. Folds over the limit: catch_0.1pct {over['catch_0.1pct'] or 'none'}, catch_rules "
    f"{over['catch_rules'] or 'none'}. **Recommendation: "
    f"{'average the 5 seeds’ scores in the final model' if AVERAGE_SEEDS else 'a single seed'}.**", "",
    "Tuned lead over the default (means over seeds 1-5; `lead (seed 42)` = tuned seed 42 minus Experiment 1's run A):",
    "", fmt(lead, index=False), "",
    "## Experiment 3: 1:20 undersampling vs class weights (tuned settings, seeds 1-5, paired by seed)", "",
    fmt(sample_info, index=False), "", fmt(exp3), "", "Mean of folds and seeds:", "", fmt(means), "",
    "Fit time is LightGBM's fit only, on this Mac with other jobs possibly running; compare the two columns, not the "
    "absolute seconds.", ""]
(REPORT / "report.md").write_text("\n".join(lines) + "\n")
print(f"-> {REPORT}  ({(time.time() - T0) / 60:.0f} min in all)")

22 model variants fitted and scored on the 3 folds
-> outputs/modelling/models/v1_full/stability  (50 min in all)


## Summary (written after the run)

1. **Reproducibility.** The default model is fully reproducible on one machine: two runs in a row match exactly (0
   difference in every measure and every score), and today's run equals this Mac's 26–27 Oct runs (mean weekly recall
   0.4086). The three different numbers (0.4086 / 0.4023 / 0.4123) came from three computers with three copies of
   `data/v1_full`; nudging the app score by one part in a million moves the mean to 0.4122, so last-digit differences
   between machines are enough to explain them. The gaps are about the size of the seed spread.
2. **Seed spread.** For the tuned model the SD over 5 seeds of `catch_0.1pct` (0.006–0.008) exceeds the 0.005 limit in
   all folds (`catch_rules` 0.004–0.010 stays under 0.012), so by the pre-registered rule we **recommend averaging 5
   seeds' scores** in the final model. The tuned model's lead over the default is larger than the seed spread on weekly
   recall (+0.010 to +0.044 per fold) but not on `catch_0.1pct`, `catch_rules` or `first_caught_2wk`.
3. **1:20 undersampling.** Same early detection as class weights (`catch_rules` +0.003, `first_caught_2wk` +0.006),
   slightly weaker ranking (weekly recall −0.007, PR-AUC −0.020, mostly fold 1), 2–4x faster training.

**Decisions for the team**

- Average 5 seeds in the final model (rule triggered), or keep the frozen single seed and state the seed spread in the
  report? Averaging needs a change to `src/models/train.py` and a retrain before the 4 Nov freeze.
- Produce every reported number on one machine and one data copy; optionally compare data hashes across machines.
- Should the lab's noise levels include seed spread (e.g. judge candidates on a seed average), given that the seed SD of
  `catch_0.1pct` is above its placebo noise level?
- The tuned settings' gain does not reach the rules-budget measures: keep them (weekly recall is the agreed tuning
  measure), but say so in the report.
- Report the 1:20 ablation as "no gain in early detection, slightly worse ranking, faster": class weights stay.

**Looks:** 22 model variants were fitted and scored on the v1_full folds (6 in experiment 1, 11 in experiment 2, 5 in
experiment 3); none chose a feature or a setting. On top of the lab ledger's trials and tuning step 2's 40 trials, the
folds have now been used many times: they are a development yardstick, not an unbiased estimate.